# AutoTech JPB Solutions - Agente Autónomo y Funcional de Taller
### Asignatura: ISY0101 - Ingeniería de Soluciones con IA
**Integrantes**: Paolo Jorquera, Benjamín Arriaza y José Castillo  
**Evaluación**: Evaluación Parcial N°2 - Desarrollo de un Agente Funcional de Taller Mecánico  


## 1. Instalación de Dependencias en Google Colab


In [ ]:
# Instalación de librerías en Google Colab para Agentes LangChain, RAG y Streamlit
!pip install -q langchain langchain-classic langchain-openai openai pypdf sentence-transformers faiss-cpu streamlit langchain-text-splitters reportlab langchain-groq gdown python-dotenv
print("[OK] Dependencias de LangChain y AutoTech instaladas con éxito.")

[OK] Dependencias de LangChain y AutoTech instaladas con éxito.


## 2. Descarga y Sincronización Automática de Manuales y Archivos de Taller
Descarga directa y automática desde Google Drive Compartido para que Colab disponga de los 5 manuales técnicos, scripts base y el caché vectorial.

In [ ]:
# Descarga y sincronización de manuales y archivos base desde Google Drive
import gdown, os, shutil

DRIVE_PUBLIC_URL = "https://drive.google.com/drive/folders/1UShpyjhW2rfinSafBseYJie8rw_P_dsK"
print("[Google Drive Público] Verificando manuales y archivos base...")
try:
    pdf_check = ["ficha_tecnica_fiat_palio.pdf", "autotech_vector_cache.pkl", "MANUAL-DE-OPERACIONES-DE-TALLER-.pdf"]
    if any(not os.path.exists(p) for p in pdf_check):
        gdown.download_folder(DRIVE_PUBLIC_URL, quiet=True)
        folder_name = "AutoTech_Manuales"
        if os.path.exists(folder_name):
            for fname in os.listdir(folder_name):
                src = os.path.join(folder_name, fname)
                dst = os.path.join(".", fname)
                if os.path.isfile(src):
                    if not os.path.exists(dst) or fname.endswith(".pdf") or fname.endswith(".pkl"):
                        try:
                            shutil.move(src, dst)
                        except Exception:
                            pass
            try:
                shutil.rmtree(folder_name)
            except Exception:
                pass
    print("[OK] 5 Manuales técnicos y caché vectorial listos en la raíz de Colab.")
except Exception as e:
    print(f"[Aviso] Verificación de descarga: {e}")

# Verificación de scripts principales para ejecución en Colab
scripts_requeridos = ["autotech_agent_ep2.py", "app_ep2.py", "rag_engine.py"]
faltantes = [s for s in scripts_requeridos if not os.path.exists(s)]
if not faltantes:
    print("[OK] Todos los scripts del agente y Streamlit están presentes y listos.")
else:
    print(f"[IMPORTANTE] Archivos faltantes en la sesión: {faltantes}")
    print("Por favor arrastra y suelta esos archivos en el panel izquierdo de Colab (Archivos 📁).")


[Google Drive Público] Verificando manuales y archivos base...
[OK] 5 Manuales técnicos y caché vectorial listos en la raíz de Colab.
[OK] Todos los scripts del agente y Streamlit están presentes y listos.


## 3. Carga Segura de Credenciales y Selección de Modelo (Google Colab Secrets & .env)

In [ ]:
# Carga segura de credenciales desde Google Colab Secrets (LLM_API_KEY)
import os
try:
    from google.colab import userdata
    try:
        val = userdata.get('LLM_API_KEY')
        if val:
            os.environ['LLM_API_KEY'] = str(val).strip()
    except Exception:
        pass
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

# Si no está en secrets, solicitar interactivamente como respaldo
if not os.environ.get('LLM_API_KEY'):
    import getpass
    print("Ingresa tu clave de API (o agrégala en el panel '🔑' Secretos de Colab con el nombre LLM_API_KEY):")
    k = getpass.getpass('LLM_API_KEY: ').strip()
    if k:
        os.environ['LLM_API_KEY'] = k

# Persistir credencial en .env para que Streamlit y procesos en segundo plano la lean automáticamente
api_val = os.environ.get('LLM_API_KEY', '')
with open('.env', 'w', encoding='utf-8') as f:
    f.write(f'LLM_API_KEY={api_val}\n')
    f.write('LLM_BASE_URL=https://api.groq.com/openai/v1\n')
    f.write('LLM_MODEL=openai/gpt-oss-20b\n')

os.environ.setdefault('LLM_BASE_URL', 'https://api.groq.com/openai/v1')
os.environ['LLM_MODEL'] = 'openai/gpt-oss-20b'
print(f"[OK] Secreto 'LLM_API_KEY' configurado exitosamente en el entorno y en .env")
print(f"[OK] Modelo NO-LLAMA activo: {os.environ.get('LLM_MODEL')}")


[OK] Secreto 'LLM_API_KEY' configurado exitosamente en el entorno y en .env
[OK] Modelo NO-LLAMA activo: openai/gpt-oss-20b


## 4. Inicialización del Motor de Base de Conocimiento RAG (Caché Persistente)

In [ ]:
from rag_engine import AutoTechVectorStore, CACHE_FILE_NAME
import time

t0 = time.time()
pdfs = [
    "ficha_tecnica_fiat_palio.pdf",
    "MANUAL-DE-OPERACIONES-DE-TALLER-.pdf",
    "manual_fallas_multimarca_diagnostico.pdf",
    "manual_recepcion_fallas_cliente_taller.pdf",
    "mecanica-automotriz-mantenimiento-de-motores.pdf"
]

vector_store = AutoTechVectorStore()
vector_store.build_or_load_index(pdfs)
print(f"[OK] VectorStore listo en {time.time()-t0:.2f}s | Fragmentos indexados: {len(vector_store.chunks)}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[Caché Vectorial] CARGA ULTRA-RÁPIDA COMPLETADA (553 fragmentos en < 0.1s desde 'autotech_vector_cache.pkl')
[OK] VectorStore listo en 5.06s | Fragmentos indexados: 553


## 5. Definición de Herramientas Autónomas
Implementamos la tríada de herramientas exigida en la rúbrica:
1. **Herramienta de Consulta RAG (`consulta_manual_tecnico`)**: Búsqueda semántica en los manuales de taller.
2. **Herramienta de Consulta de Stock (`consultar_inventario_repuestos`)**: Acceso directo al inventario de bodega.
3. **Herramienta de Escritura y Persistencia (`registrar_orden_reparacion`)**: Formaliza y guarda órdenes de trabajo en formato estructurado JSON.

In [ ]:
from autotech_agent_ep2 import (
    consulta_manual_tecnico,
    consultar_inventario_repuestos,
    registrar_orden_reparacion,
    buscar_en_internet,
    AUTOTECH_TOOLS
)

print(f"Herramientas autónomas configuradas para el Agente ({len(AUTOTECH_TOOLS)}):\n")
for idx, t in enumerate(AUTOTECH_TOOLS, 1):
    print(f"{idx}. {t.name}: {t.description[:85]}...")


Herramientas autónomas configuradas para el Agente (4):

1. consulta_manual_tecnico: Herramienta de CONSULTA SEMÁNTICA RAG en los 5 manuales de taller mecánico de AutoTec...
2. consultar_inventario_repuestos: Herramienta de CONSULTA DE INVENTARIO Y STOCK en la bodega de AutoTech JPB Solutions....
3. registrar_orden_reparacion: Herramienta de ESCRITURA Y PERSISTENCIA. Registra o actualiza una Orden de Trabajo of...
4. buscar_en_internet: Herramienta de BÚSQUEDA TÉCNICA EXTERNA EN PORTALES ESPECIALIZADOS DE AUTOMOCIÓN.
Con...


## 6. Configuración del Agente de LangChain y Memoria Conversacional


In [ ]:
from autotech_agent_ep2 import create_autotech_agent, AutoTechSessionMemory

agent_executor = create_autotech_agent()
session_memory = AutoTechSessionMemory(max_history_turns=6)
print(f"[OK] Agente Funcional AutoTech (Modelo: {os.environ.get('LLM_MODEL')}) y Memoria Conversacional listos.")


[OK] Agente Funcional AutoTech (Modelo: openai/gpt-oss-20b) y Memoria Conversacional listos.


## 7. Batería de Pruebas: Demostración de Autonomía, Memoria y Toma de Decisiones
Ejecutamos un flujo realista de 3 etapas en el taller automotriz:
- **Etapa 1**: Consulta técnica RAG de manual (falla de frenos y lubricación de Fiat Palio).
- **Etapa 2**: Verificación de inventario recordando el vehículo previo en memoria.
- **Etapa 3**: Generación autónoma y persistencia de Orden de Trabajo formal.

In [ ]:
from autotech_agent_ep2 import run_ep2_test_battery

# Ejecución interactiva de las pruebas del agente
run_ep2_test_battery()

DEMOSTRACIÓN EVALUACIÓN PARCIAL 2 - AGENTE NO-LLAMA (OPENAI/GPT-OSS-20B)

--- DEMOSTRACIÓN DE PLANIFICACIÓN PREVIA (PLAN-AND-EXECUTE / IE5) ---
Objetivo: Recepcionar un Fiat Palio 1.3 con chirrido severo en frenos y nivel bajo de lubricante, verificar stock de repuestos y formalizar orden de trabajo.

Plan de Trabajo Generado por el Planificador:
**Plan Secuencial y Priorizado de Atención – Fiat Palio 1.3**

| Paso | Acción concreta | Herramienta | Justificación técnica |
|------|-----------------|-------------|-----------------------|
| **[Paso 1]** | **Inspección inicial y diagnóstico de frenos** – Realizar prueba de frenado, revisar pastillas, discos y nivel de líquido de frenos. | `consulta_manual_tecnico` (manual de Fiat Palio 1.3) | El chirrido severo indica desgaste de pastillas o fricción anómala; la regla de prioridad alta exige evaluar inmediatamente la seguridad del vehículo. |
| **[Paso 2]** | **Verificar nivel y calidad del lubricante** – Medir nivel de líquido de frenos y

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[Caché Vectorial] CARGA ULTRA-RÁPIDA COMPLETADA (553 fragmentos en < 0.1s desde 'autotech_vector_cache.pkl')
Manuales AutoTech para 'Aceite recomendado y capacidad de aceite para Fiat Palio 1.3 Fire 2008 con cambio de filtro; diagnóstico de frenos con ruido y freno largo':
- [ficha_tecnica_fiat_palio.pdf | Pág. 1]: =================================================================== FICHA TÉCNICA Y ESPECIFICACIONES DE TALLER - FIAT PALIO 1.3 8V (FIRE) =================================================================== [INFORMACIÓN GENERAL DEL VEHÍCULO] - Modelo: Fiat Palio Young / SX (Año 2004) - Motor: 1.3 Lit...
- [ficha_tecnica_fiat_palio.pdf | Pág. 2]: [SISTEMA DE FRENOS Y NEUMÁTICOS]  - Frenos delanteros: Discos sólidos (Medida estándar).  - Frenos traseros: Tambor con zapatas autoajustables.  - Presión de neumáticos recomendada: 28 PSI (Uso normal) / 31 PSI (Carga completa  o carretera).  - Medida de llantas/neumáticos de fábrica: 165/70 R13 o 1...

Invoking: `consulta_manual_tecn

## 8. Verificación de Persistencia de la Orden de Trabajo Generada
Revisamos el archivo JSON generado por la herramienta de escritura (`ordenes_trabajo_autotech.json`) para evidenciar la correcta persistencia del estado.

In [ ]:
import json

if os.path.exists("ordenes_trabajo_autotech.json"):
    with open("ordenes_trabajo_autotech.json", "r", encoding="utf-8") as f:
        ordenes = json.load(f)
    print(f"Total de órdenes registradas: {len(ordenes)}\n")
    print(json.dumps(ordenes[-1], indent=2, ensure_ascii=False))
else:
    print("Aún no se han generado órdenes de trabajo.")

Total de órdenes registradas: 5

{
  "id_orden": "OT-1793244662",
  "fecha_ingreso": "2026-10-29 00:31:02",
  "fecha_cita_programada": "30-10-2026 18:00 hrs (6:00 pm)",
  "cliente": "Benjamín Arriaza",
  "vehiculo": "Fiat Palio 1.3 Fire 2008",
  "prioridad": "Alta",
  "diagnostico": "Frenos delanteros con pastillas desgastadas y ruido de chillido. Servicio de cambio de pastillas, limpieza de calipers y purgado DOT4.",
  "repuestos": [
    {
      "repuesto": "Juego Pastillas de Freno Delanteras Fiat/Universal",
      "precio_clp": 22990,
      "stock_actual": 15,
      "bodega": "Pasillo C - Estante 3"
    },
    {
      "repuesto": "Líquido de Frenos DOT 4 (500ml)",
      "precio_clp": 4990,
      "stock_actual": 30,
      "bodega": "Pasillo A - Estante 4"
    }
  ],
  "costo_repuestos_clp": 27980,
  "costo_mano_obra_clp": 30000,
  "desglose_mano_obra": [
    {
      "tarea": "Cambio de juego de pastillas de freno delanteras y limpieza de calipers",
      "costo_clp": 20000
    },
   

## 9. Despliegue de Interfaz Web Streamlit con Cloudflare Tunnel


In [ ]:
# Despliegue de Interfaz Web Streamlit con Cloudflare Tunnel en Google Colab
import subprocess, time, re, os

# 1. Asegurar sincronización de la clave LLM_API_KEY
key_val = os.environ.get('LLM_API_KEY', '')
if key_val:
    with open('.env', 'w', encoding='utf-8') as f:
        f.write(f'LLM_API_KEY={key_val}\nLLM_BASE_URL=https://api.groq.com/openai/v1\nLLM_MODEL=openai/gpt-oss-20b\n')

# 2. Detener instancias previas
!pkill -f streamlit 2>/dev/null || true
!pkill -f cloudflared 2>/dev/null || true

# 3. Iniciar Streamlit en segundo plano vinculando LLM_API_KEY
print("[1/2] Iniciando servidor Streamlit en puerto 8501 con credencial 'LLM_API_KEY' vinculada...")
env_cmd = f'LLM_API_KEY="{key_val}" nohup streamlit run app_ep2.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &'
subprocess.Popen(env_cmd, shell=True)
time.sleep(3)

# 4. Abrir túnel público con Cloudflare
print("[2/2] Abriendo túnel seguro de Cloudflare...")
tunnel_proc = subprocess.Popen(
    'npx -y cloudflared tunnel --url http://localhost:8501',
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for _ in range(35):
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(1)
        continue
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print('\n' + '=' * 75)
    print(f' ENLACE PÚBLICO: {public_url}')
    print('=' * 75 + '\n')
    from IPython.display import display, HTML
    display(HTML(f'''<div style="padding: 15px; background: #0f172a; border: 2px solid #06b6d4; border-radius: 10px;">
        <h3 style="color: #f8fafc; margin: 0;"> AutoTech Web App Activa</h3>
        <p style="color: #94a3b8; margin: 5px 0 10px 0;">Haz clic en el enlace para abrir la aplicación web en tu navegador:</p>
        <a href="{public_url}" target="_blank" style="color: #38bdf8; font-size: 1.2rem; font-weight: bold; text-decoration: underline;"> {public_url}</a>
    </div>'''))
else:
    print('\n[Aviso] Conectando túnel... Si tarda unos segundos, revisa el registro a continuación:')
    try:
        for line in tunnel_proc.stdout:
            print(line, end='')
    except KeyboardInterrupt:
        print('\nTúnel detenido.')


^C
^C
[1/2] Iniciando servidor Streamlit en puerto 8501 con credencial 'LLM_API_KEY' vinculada...
[2/2] Abriendo túnel seguro de Cloudflare...

 ENLACE PÚBLICO: https://tells-drive-writer-ski.trycloudflare.com

